# Simple log(gf) tweak and line-profile fit

This notebook keeps things minimal:
1) change log(gf) and measure delta flux
2) fit log(gf) to a provided line profile

Note: VALD linelist wavelengths are vacuum. We pick the line center from the linelist to avoid air/vacuum mismatches.

This version loads the observed solar spectrum from `data/obs_spec.txt (via JORG_DATA_DIR)` and interpolates it onto the synthesis grid.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter1d

# Add src to path for editable installs (optional)
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    src_path = repo_root / "src"
    if (src_path / "jorg").exists():
        sys.path.insert(0, str(src_path))
        data_dir = repo_root / "data"
        if data_dir.exists():
            os.environ.setdefault("JORG_DATA_DIR", str(data_dir))
        break
    repo_root = repo_root.parent


In [ ]:
# Basic setup
Teff = 5772
logg = 4.4
m_H = 0.0
vmic = 1.09
hydrogen_lines = False
cntm_step = 1.0

atm = interpolate_marcs(Teff, logg, m_H)
A_X = create_korg_compatible_abundance_array(m_H)

# Full wavelength window for comparison
wl_min, wl_max = 5000.0, 5200.0

from jorg.data import get_data_path

linelist_path = get_data_path("vald_extract_stellar_solar_threshold001.vald", must_exist=False)
if not linelist_path.exists():
    raise FileNotFoundError("VALD linelist not found. Set JORG_DATA_DIR to your data bundle.")
linelist = read_linelist(str(linelist_path), format='vald')

# Pick a line close to your target wavelength
# target_wl = 5001.6
target_wl = 5180.25
line_wls = linelist.wavelengths_angstrom()
line_idx = int(np.argmin(np.abs(line_wls - target_wl)))
line_center = line_wls[line_idx]
line = linelist[line_idx]
line_label = f"{line.species} @ {line_center:.4f} A"
print(f'Using line_center = {line_center:.4f} Angstrom')

# Load observed solar spectrum and align synthesis grid to observed resolution
obs_path = get_data_path("obs_spec.txt", must_exist=False)
if not obs_path.exists():
    raise FileNotFoundError("Observed spectrum not found. Set JORG_DATA_DIR to your data bundle.")
obs_lines = obs_path.read_text().splitlines()
obs_wl = np.fromstring(obs_lines[0], sep=' ')
obs_flux_raw = np.fromstring(obs_lines[1], sep=' ')
finite = np.isfinite(obs_flux_raw)
obs_wl = obs_wl[finite]
obs_flux_raw = obs_flux_raw[finite]

# obs_spec.txt is in air wavelengths; convert to vacuum to match VALD/Jorg
obs_wavelengths = 'air'
if obs_wavelengths == 'air':
    obs_wl = air_to_vacuum(obs_wl * 1e-8) * 1e8

obs_cont = np.nanpercentile(obs_flux_raw, 99.5)
obs_flux = obs_flux_raw / obs_cont

obs_mask = (obs_wl >= wl_min) & (obs_wl <= wl_max)
wavelength_grid = obs_wl[obs_mask]
observed_flux = obs_flux[obs_mask]
if wavelength_grid.size == 0:
    raise ValueError('Observed spectrum has no points in the requested window.')
print(f'Observed spectrum: {len(obs_wl)} points, continuum~{obs_cont:.3f} ({obs_wavelengths}->vacuum)')
print(f'Using observed grid: {len(wavelength_grid)} points from {wavelength_grid[0]:.2f}-{wavelength_grid[-1]:.2f} A')


def gaussian_convolve(wavelengths, flux, fwhm_angstrom):
    spacing = float(np.median(np.diff(wavelengths)))
    if spacing <= 0:
        raise ValueError('Wavelength grid must be strictly increasing for convolution.')
    sigma_angstrom = fwhm_angstrom / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    sigma_pix = sigma_angstrom / spacing
    return gaussian_filter1d(flux, sigma_pix, mode='nearest')


# Approximate instrumental FWHM from observed sampling and optionally reduce resolution.
pixels_per_fwhm = 2.5
sampling_fwhm = pixels_per_fwhm * float(np.median(np.diff(wavelength_grid)))

# Reduce resolution to better match observed spectra (lower R => broader lines).
target_R = 35000.0  # set to None to keep sampling-based resolution
if target_R is None:
    instrument_fwhm = sampling_fwhm
else:
    instrument_fwhm = max(sampling_fwhm, line_center / float(target_R))

instrument_R = line_center / instrument_fwhm
print(f'Instrument FWHM ~{instrument_fwhm:.4f} A (R~{instrument_R:.0f}, {pixels_per_fwhm:.1f} px/FWHM, target R={target_R})')


def apply_instrument_broadening(flux, wavelengths):
    return gaussian_convolve(wavelengths, flux, instrument_fwhm)


# Synthesize spectrum in one call (lines + continuum) on observed grid
base_result = synthesize(
    atm,
    linelist,
    A_X,
    wavelengths=wavelength_grid,
    hydrogen_lines=hydrogen_lines,
    verbose=False,
    logg=logg,
    cntm_step=cntm_step,
    vmic=vmic,
)

base_flux = np.asarray(base_result.flux) / np.asarray(base_result.cntm)
base_flux_conv = apply_instrument_broadening(base_flux, wavelength_grid)

# Local fit window (keeps optimization fast)
fit_window_half_width = 0.25
fit_wl_min = line_center - fit_window_half_width
fit_wl_max = line_center + fit_window_half_width
fit_line_buffer = 10.0

fit_linelist = linelist.filter_by_wavelength(
    fit_wl_min - fit_line_buffer,
    fit_wl_max + fit_line_buffer,
    unit='angstrom'
)
fit_mask = (wavelength_grid >= fit_wl_min) & (wavelength_grid <= fit_wl_max)
fit_wavelength_grid = wavelength_grid[fit_mask]
observed_flux_fit = observed_flux[fit_mask]
print(f"Fit window: {fit_wl_min:.3f}-{fit_wl_max:.3f} A, {len(fit_linelist)} lines")


In [ ]:
# Compare full-window spectra: Korg vs Jorg vs observed (optional)
import os
from pathlib import Path

korg_script = os.environ.get("KORG_JL_SCRIPT")
if korg_script is None:
    korg_repo = os.environ.get("KORG_JL_REPO")
    if korg_repo:
        korg_script = str(Path(korg_repo) / "korg_compare_params.jl")

out_dir = Path("comparison_outputs")
out_dir.mkdir(exist_ok=True)

tag = f"Teff{Teff:.0f}_g{logg:.2f}_mH{m_H:.2f}_wl{wl_min:.0f}-{wl_max:.0f}"
tag = tag.replace('.', 'p').replace('-', 'm')

k_rect_conv = None
if not korg_script:
    print("Skipping Korg comparison. Set KORG_JL_SCRIPT or KORG_JL_REPO to enable.")
else:
    julia_bin = os.environ.get("JULIA_BIN", "julia")
    korg_script_path = Path(korg_script)
    if not korg_script_path.exists():
        raise FileNotFoundError(f"Korg script not found: {korg_script_path}")

    cmd = [
        julia_bin,
        f"--project={korg_script_path.parent}",
        str(korg_script_path),
        str(Teff),
        str(logg),
        str(m_H),
        str(wl_min),
        str(wl_max),
        tag,
        str(out_dir),
    ]
    subprocess.run(cmd, check=True)

    k_lines = np.loadtxt(out_dir / f"korg_{tag}_spectrum_with_lines.txt")
    k_wl, k_flux, k_cntm, k_rect = k_lines.T
    k_rect_interp = np.interp(wavelength_grid, k_wl, k_rect)
    k_rect_conv = apply_instrument_broadening(k_rect_interp, wavelength_grid)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=wavelength_grid, y=base_flux_conv, mode='lines', name='jorg'))
if k_rect_conv is not None:
    fig.add_trace(go.Scatter(x=wavelength_grid, y=k_rect_conv, mode='lines', name='korg'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=observed_flux, mode='lines', name='observed'))
fig.update_layout(
    template='plotly_white',
    width=900,
    height=450,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Rectified Flux',
    title=f'Observed vs Jorg vs Korg (line: {line_label})'
)
fig


In [ ]:
# Find Fe I lines with large log(gf) uncertainty (depth-based proxy)
from jorg.statmech.species import Species

fe1 = Species.from_string('Fe I')
depth_window = 0.10
all_wl = np.asarray(linelist.wavelengths_angstrom())
all_wl_sorted = np.sort(all_wl)

rows = []
for line in linelist:
    wl = float(line.wavelength) * 1e8
    if wl < wl_min or wl > wl_max:
        continue
    if line.species != fe1:
        continue
    mask = (wavelength_grid >= wl - depth_window) & (wavelength_grid <= wl + depth_window)
    if np.count_nonzero(mask) < 3:
        continue
    model_min = float(np.min(base_flux_conv[mask]))
    obs_min = float(np.min(observed_flux[mask]))
    model_depth = max(0.0, 1.0 - model_min)
    obs_depth = max(0.0, 1.0 - obs_min)
    depth = max(model_depth, obs_depth)
    uncertainty_proxy = 1.0 / max(depth, 1e-3)

    left = int(np.searchsorted(all_wl_sorted, wl - depth_window))
    right = int(np.searchsorted(all_wl_sorted, wl + depth_window, side='right'))
    blend_count = max(0, right - left - 1)

    rows.append((wl, float(line.log_gf), model_depth, obs_depth, blend_count, uncertainty_proxy))

rows.sort(key=lambda r: r[-1], reverse=True)
top_n = 15
top_rows = rows[:top_n]

table = go.Figure(data=[go.Table(
    header=dict(values=['Wavelength (A)', 'log(gf)', 'model depth', 'obs depth', 'blend count', 'uncertainty proxy'],
                align='left'),
    cells=dict(values=[
        [f'{r[0]:.4f}' for r in top_rows],
        [f'{r[1]:.3f}' for r in top_rows],
        [f'{r[2]:.4f}' for r in top_rows],
        [f'{r[3]:.4f}' for r in top_rows],
        [f'{r[4]:d}' for r in top_rows],
        [f'{r[5]:.2f}' for r in top_rows],
    ], align='left'))
])
table.update_layout(
    template='plotly_white',
    width=900,
    height=420,
    title='Fe I lines with large log(gf) uncertainty (proxy) in 5000-5200 A'
)
table.show()


In [ ]:
# Change log(gf) and compute delta flux
delta_loggf = 0.5

modifier = LogGFModifier(linelist, wavelength_tolerance=0.01)
modifier.adjust_line(line_center, delta_loggf=delta_loggf)
modified_linelist = modifier.apply_modifications()

modified_result = synthesize(
    atm,
    modified_linelist,
    A_X,
    wavelengths=wavelength_grid,
    hydrogen_lines=hydrogen_lines,
    verbose=False,
    logg=logg,
    cntm_step=cntm_step,
    vmic=vmic,
    use_chemical_equilibrium_from=base_result,
)

modified_flux = np.asarray(modified_result.flux) / np.asarray(modified_result.cntm)
modified_flux_conv = apply_instrument_broadening(modified_flux, wavelength_grid)
delta_flux = modified_flux_conv - base_flux_conv
print(f'Delta flux: min={delta_flux.min():.3e}, max={delta_flux.max():.3e}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=wavelength_grid, y=base_flux_conv, mode='lines', name='base'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=modified_flux_conv, mode='lines', name='modified'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=delta_flux, mode='lines', name='delta', line=dict(dash='dash')))
fig.add_vline(x=line_center, line_dash='dot', line_color='gray', line_width=1)
fig.add_annotation(
    x=line_center,
    y=float(np.max(base_flux_conv)),
    text=line_label,
    showarrow=False,
    textangle=90,
    xanchor='left',
    yanchor='bottom'
)
fig.update_layout(
    template='plotly_white',
    width=800,
    height=400,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Normalized flux',
)
fig.show()


In [ ]:
# Fit log(gf) to a given line profile using BFGS - OPTIMIZED with LogGFFitter
# observed_flux is loaded from obs_spec.txt

if fit_wavelength_grid.size == 0:
    raise ValueError("Fit wavelength grid is empty. Check fit_window_half_width.")
print(f"Fitting window: ±{fit_window_half_width} A around {line_center:.4f} A")

# === NEW OPTIMIZED APPROACH (2-5x faster) ===
# LogGFFitter caches continuum calculations, avoiding redundant CE and opacity computation
from jorg.fit import LogGFFitter

print("🚀 Using optimized LogGFFitter (caches continuum calculations)...")
fitter = LogGFFitter(
    atm=atm,
    A_X=A_X,
    wavelengths=fit_wavelength_grid,
    linelist=fit_linelist,
    vmic=vmic,
    cntm_step=cntm_step,
    hydrogen_lines=hydrogen_lines,
    verbose=False
)

# Fast fitting - all continuum calculations cached internally
result = fitter.fit_line(
    line_center=line_center,
    observed_flux=observed_flux_fit,
    postprocess_flux=apply_instrument_broadening,
    method='BFGS',
    gtol=1e-4, maxiter=20
)

print(f'Best delta_loggf = {result.best_delta_loggf:.3f} (success={result.success})')
print(f'Final loggf = {result.best_loggf:.3f}')
print(f'Evaluations: {result.n_evaluations}')

# The result object contains the best-fit spectrum
best_flux = result.best_fit_flux

fig = go.Figure()
fig.add_trace(go.Scatter(x=fit_wavelength_grid, y=observed_flux_fit, mode='lines', name='observed'))
fig.add_trace(go.Scatter(x=fit_wavelength_grid, y=best_flux, mode='lines', name='best-fit'))
fig.add_vline(x=line_center, line_dash='dot', line_color='gray', line_width=1)
fig.add_annotation(
    x=line_center,
    y=float(np.max(observed_flux_fit)),
    text=line_label,
    showarrow=False,
    textangle=90,
    xanchor='left',
    yanchor='bottom'
)
fig.update_layout(
    template='plotly_white',
    width=800,
    height=400,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Normalized flux',
    title=f'Optimized LogGF Fitting (cached continuum for ~3x speedup)'
)
fig.show()

# === COMPARISON: OLD vs NEW APPROACH ===
print("\n📊 API Comparison:")
print("  OLD: objective() → synthesize() each iteration (slow)")
print("  NEW: LogGFFitter.fit_line() → resynthesize_from_continuum() (fast)")
print("\n  The new API caches:")
print("    - Chemical equilibrium (number densities)")
print("    - Continuum opacity (independent of loggf)")
print("    - Source function (Planck B_λ(T))")
print("  And only recalculates:")
print("    - Line opacity (depends on loggf)")
print("    - Radiative transfer")

# === BONUS: Quick interactive exploration ===
print(f"\n💡 Tip: You can now quickly explore different loggf values:")
print("   flux = fitter.synthesize_with_loggf(0.1, line_center)")
print("   # Returns flux for loggf + 0.1 dex (uses cached continuum!)")


In [ ]:
# === BENCHMARK: OLD vs NEW APPROACH ===
# Compare runtime of unoptimized vs optimized fitting

import time

print("📊 BENCHMARKING: Old vs New Approach")
print("=" * 60)

# OLD APPROACH (unoptimized)
print("")
print("1. OLD APPROACH: synthesize() each iteration")
print("   (calculates continuum every time)")

def objective_old(x):
    """Old approach: full synthesis each iteration"""
    delta = float(x[0])
    modifier = LogGFModifier(fit_linelist, wavelength_tolerance=0.01)
    modifier.adjust_line(line_center, delta_loggf=delta)
    candidate_linelist = modifier.apply_modifications()

    result = synthesize(
        atm, candidate_linelist, A_X,
        wavelengths=fit_wavelength_grid,
        hydrogen_lines=hydrogen_lines,
        verbose=False,
        logg=logg,
        cntm_step=cntm_step,
        vmic=vmic,
        use_chemical_equilibrium_from=base_result,  # Only CE is cached
    )
    model_flux = np.asarray(result.flux) / np.asarray(result.cntm)
    if model_flux.shape != fit_wavelength_grid.shape:
        model_flux = np.interp(fit_wavelength_grid, result.wavelengths, model_flux)
    model_flux = apply_instrument_broadening(model_flux, fit_wavelength_grid)
    return float(np.mean((model_flux - observed_flux_fit) ** 2))

# Time the old approach (just 3 iterations for speed comparison)
x0 = np.array([0.0])
t0 = time.time()
for i in range(3):
    obj_val = objective_old(x0)
old_time = time.time() - t0
print(f"   Time for 3 iterations: {old_time:.2f}s")

# NEW APPROACH (optimized)
print("")
print("2. NEW APPROACH: LogGFFitter with cached continuum")
print("   (continuum calculated once, reused)")

t0 = time.time()
for i in range(3):
    # Fast synthesis using cached continuum
    flux_test = apply_instrument_broadening(
        fitter.synthesize_with_loggf(0.0, line_center), fit_wavelength_grid
    )
new_time = time.time() - t0
print(f"   Time for 3 iterations: {new_time:.2f}s")

# Calculate speedup
speedup = old_time / new_time
print("")
print(f"🚀 SPEEDUP: {speedup:.1f}x faster")
print(f"   (varies with wavelength range and iteration count)")

# Estimate full fitting time
old_fit_time = old_time * 7  # ~20 iterations / 3
new_fit_time = new_time * 7
print("")
print("Estimated full fitting time:")
print(f"  OLD: {old_fit_time:.1f}s")
print(f"  NEW: {new_fit_time:.1f}s")
print(f"  Time saved: {old_fit_time - new_fit_time:.1f}s")


## Summary: Optimized LogGF Fitting

### New Features (January 2026)

| Feature | Description |
|---------|-------------|
| **`LogGFFitter`** | High-level class for optimized loggf fitting with continuum caching |
| **`fit_loggf_quick()`** | Convenience function for quick one-off fits |
| **`resynthesize_from_continuum()`** | Low-level function for efficient resynthesis |
| **`alpha_continuum`** | New field in `SynthesisResult` for caching continuum opacity |
| **`source_function`** | New field in `SynthesisResult` for caching Planck function |

### Performance Improvement

- **2-5x speedup** for loggf fitting
- Achieved by caching continuum calculations that are independent of loggf
- Chemical equilibrium was already cached via `use_chemical_equilibrium_from`
- Now also caching: continuum opacity + source function

### API Reference

```python
# Approach 1: Direct class usage (recommended for multiple lines)
from jorg.fit import LogGFFitter

fitter = LogGFFitter(atm, A_X, wavelengths, linelist)
result = fitter.fit_line(line_center, observed_flux)

# Quick exploration
flux = fitter.synthesize_with_loggf(0.1, line_center)  # +0.1 dex

# Approach 2: Convenience function (for quick one-off fits)
from jorg.fit import fit_loggf_quick

result = fit_loggf_quick(Teff, logg, m_H, wavelengths, linelist,
                        line_center, observed_flux)
```

### What Gets Cached?

✅ **Cached** (independent of loggf):
- Chemical equilibrium (number densities, electron densities)
- Continuum opacity (`alpha_continuum`)
- Source function (Planck B_λ(T))

⚡ **Recalculated** (depends on loggf):
- Line opacity
- Radiative transfer


In [ ]:
# === ALTERNATIVE: Convenience function for quick fitting ===
# fit_loggf_quick() handles atmosphere and abundance setup automatically

print("⚡ QUICK FIT: Using fit_loggf_quick() convenience function")
print("=" * 60)

from jorg.fit import fit_loggf_quick

# This function automatically:
# - Interpolates atmosphere from Teff, logg, m_H
# - Creates abundance array
# - Initializes LogGFFitter
# - Performs the fit

quick_result = fit_loggf_quick(
    Teff=Teff,
    logg=logg,
    m_H=m_H,
    wavelengths=fit_wavelength_grid,
    linelist=fit_linelist,
    line_center=line_center,
    observed_flux=observed_flux_fit,
    postprocess_flux=apply_instrument_broadening,
    vmic=vmic,
    cntm_step=cntm_step,
    hydrogen_lines=hydrogen_lines,
    method='BFGS'
)

print(f"\n📊 Quick Fit Results:")
print(f"  Best delta_loggf: {quick_result.best_delta_loggf:.4f}")
print(f"  Final loggf: {quick_result.best_loggf:.4f}")
print(f"  Success: {quick_result.success}")
print(f"  Evaluations: {quick_result.n_evaluations}")

print("\n💡 When to use fit_loggf_quick():")
print("  - Quick one-off fits without setup code")
print("  - Interactive sessions")
print("  - Prototyping and testing")
print("\n  When to use LogGFFitter directly:")
print("  - Fitting multiple lines with same stellar parameters")
print("  - Batch processing")
print("  - Need synthesize_with_loggf() for exploration")


In [ ]:
# === INTERACTIVE EXPLORATION with synthesize_with_loggf ===
# Quickly explore how different loggf values affect the line profile

print("🔬 Interactive Exploration: Line Profile vs log(gf)")
print("=" * 60)

# Try different loggf adjustments
deltas = [-0.3, -0.1, 0.0, 0.1, 0.3, 0.5]

fig = go.Figure()
# Add observed spectrum
fig.add_trace(go.Scatter(
    x=fit_wavelength_grid, 
    y=observed_flux_fit, 
    mode='lines', 
    name='observed',
    line=dict(color='black', width=2)
))

# Generate spectra for different loggf values
colors = ['blue', 'cyan', 'green', 'orange', 'red', 'purple']
for delta, color in zip(deltas, colors):
    flux = apply_instrument_broadening(
        fitter.synthesize_with_loggf(delta, line_center), fit_wavelength_grid
    )
    label = f'loggf +{delta:+.1f}' if delta != 0 else 'original'
    fig.add_trace(go.Scatter(
        x=fit_wavelength_grid,
        y=flux,
        mode='lines',
        name=label,
        line=dict(color=color, dash='dash' if delta != 0 else 'solid')
    ))

fig.add_vline(x=line_center, line_dash='dot', line_color='gray', line_width=1)
fig.update_layout(
    template='plotly_white',
    width=900,
    height=500,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Normalized flux',
    title='Interactive: Line Profile vs log(gf) Adjustment',
    hovermode='x unified'
)
fig.show()

print("\n💡 Key observations:")
print("  - Higher loggf → deeper absorption line")
print("  - Lower loggf → shallower absorption line")
print("  - Changes are linear in log space (0.1 dex ≈ 26% strength change)")
print("\n  This makes it easy to manually tune loggf to match observations!")
